In [16]:
from datasets import load_dataset

print("Loading product dataset...")

dataset = load_dataset(
    "ashraq/fashion-product-images-small",
    split="train[:10]"
)

print("Dataset loaded successfully!")
print("Number of products:", len(dataset))
print("Available columns:", dataset.column_names)

print("\nFIRST 3 PRODUCTS")
print("=" * 50)

for product in dataset.select(range(3)):
    print("ID:", product["id"])
    print("Name:", product["productDisplayName"])
    print("Category:", product["masterCategory"])
    print("Type:", product["articleType"])
    print("Color:", product["baseColour"])
    print("Image available:", product["image"] is not None)
    print("-" * 50)

    from pathlib import Path

# Create a folder to store product images
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)

# Get the first product
product = dataset[0]

# Get its image
image = product["image"]

# Save the image
image_path = images_dir / "product_15970.png"
image.save(image_path)

print("\nImage saved successfully!")
print("Image path:", image_path)

# Prepare product information for AI
product = dataset[0]

product_info = {
    "id": product["id"],
    "name": product["productDisplayName"],
    "category": product["masterCategory"],
    "subcategory": product["subCategory"],
    "type": product["articleType"],
    "color": product["baseColour"],
    "gender": product["gender"],
    "usage": product["usage"]
}

print("\nPRODUCT INFORMATION FOR AI")
print("=" * 50)

for key, value in product_info.items():
    print(f"{key}: {value}")

    import base64

# Open the saved image in binary mode
with open("product_images/product_15970.png", "rb") as image_file:
    image_bytes = image_file.read()

# Convert image bytes to Base64
base64_image = base64.b64encode(image_bytes).decode("utf-8")

print("\nIMAGE ENCODING")
print("=" * 50)
print("Image converted to Base64 successfully!")
print("Base64 length:", len(base64_image))
print("First 100 characters:", base64_image[:100])

import json

prompt = f"""
You are an expert e-commerce copywriter.

Create a professional product listing using the product
information below and the product image.

PRODUCT INFORMATION:
Name: {product_info['name']}
Category: {product_info['category']}
Subcategory: {product_info['subcategory']}
Type: {product_info['type']}
Color: {product_info['color']}
Gender: {product_info['gender']}
Usage: {product_info['usage']}

REQUIREMENTS:
1. Create an attractive product title.
2. Write a detailed product description between 150 and 200 words.
   This word count is mandatory.
   Before returning the final JSON, verify that the description
   contains at least 150 words and no more than 200 words.
   If the description is too short, expand it with useful,
   accurate product information without inventing details.
3. Include 5-7 key product features.
4. Generate 10-15 SEO keywords.
5. Use a professional and persuasive tone.
6. Do not invent materials, specifications, or other
   details that cannot be verified from the data or image.
7. Do not invent a price.

Return ONLY valid JSON with this structure:

{{
    "title": "Product title",
    "description": "Product description",
    "features": ["Feature 1", "Feature 2"],
    "keywords": ["Keyword 1", "Keyword 2"]
}}
"""

print("\nPRODUCT LISTING PROMPT")
print("=" * 50)
print(prompt)

Loading product dataset...
Dataset loaded successfully!
Number of products: 10
Available columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

FIRST 3 PRODUCTS
ID: 15970
Name: Turtle Check Men Navy Blue Shirt
Category: Apparel
Type: Shirts
Color: Navy Blue
Image available: True
--------------------------------------------------
ID: 39386
Name: Peter England Men Party Blue Jeans
Category: Apparel
Type: Jeans
Color: Blue
Image available: True
--------------------------------------------------
ID: 59263
Name: Titan Women Silver Watch
Category: Accessories
Type: Watches
Color: Silver
Image available: True
--------------------------------------------------

Image saved successfully!
Image path: product_images/product_15970.png

PRODUCT INFORMATION FOR AI
id: 15970
name: Turtle Check Men Navy Blue Shirt
category: Apparel
subcategory: Topwear
type: Shirts
color: Navy Blue
gender: Men
usage: Casual

IM

In [17]:
# Prepare the message with text and image
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": prompt
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{base64_image}"
                }
            }
        ]
    }
]

print("\nAPI MESSAGE PREPARATION")
print("=" * 50)
print("Message prepared successfully!")
print("Number of messages:", len(messages))
print("Content types:", [
    item["type"] for item in messages[0]["content"]
])


API MESSAGE PREPARATION
Message prepared successfully!
Number of messages: 1
Content types: ['text', 'image_url']


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Read the API key
api_key = os.getenv("OPENAI_ADMIN_KEY")

# Check that the key exists
if not api_key:
    raise ValueError("OPENAI_ADMIN_KEY not found in .env")

# Initialize the client with the API key
client = OpenAI(api_key=api_key)

print("\nCALLING OPENAI API")
print("=" * 50)

try:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        response_format={"type": "json_object"},
        max_tokens=1200
    )

    generated_listing = response.choices[0].message.content

    # Save the AI response to a JSON file
with open("first_product_listing.json", "w", encoding="utf-8") as file:
    file.write(generated_listing)

print("Product listing saved successfully!")

    print("API call successful!")
    print("\nGENERATED PRODUCT LISTING:")
    print(generated_listing)

except Exception as e:
    print("API call failed!")
    print("Error type:", type(e).__name__)
    print("Error message:", str(e))


CALLING OPENAI API
API call successful!

GENERATED PRODUCT LISTING:
{
    "title": "Turtle Check Men Navy Blue Casual Shirt",
    "description": "Elevate your casual wardrobe with the Turtle Check Men Navy Blue Shirt, an essential addition for the modern man. This stylish shirt combines contemporary design with comfort, making it suitable for a variety of occasions, from a casual outing to a relaxed weekend brunch. The rich navy blue color is not only versatile, but it also complements numerous styles, allowing you to pair it with jeans, chinos, or shorts effortlessly. Crafted for the fashion-forward man, this shirt features a classic check pattern, adding a touch of sophistication to your everyday look. The breathable fabric ensures comfort throughout the day, while the short sleeves make it perfect for warm weather. Whether you're dressing up or down, this shirt's timeless appeal will make it a favorite in your collection. Make a bold statement and express your unique style with the

In [21]:
import json

# Save the previously generated product listing
with open("first_product_listing.json", "w", encoding="utf-8") as file:
    json.dump(listing, file, indent=4, ensure_ascii=False)

print("Product listing saved successfully!")

Product listing saved successfully!


In [19]:
import json

print("\nVALIDATING AI RESPONSE")
print("=" * 50)

try:
    listing = json.loads(generated_listing)

    print("Valid JSON: True")
    print("Title:", listing.get("title"))

    description = listing.get("description", "")
    features = listing.get("features", [])
    keywords = listing.get("keywords", [])

    print("Description words:", len(description.split()))
    print("Number of features:", len(features))
    print("Number of keywords:", len(keywords))

    print("\nREQUIREMENTS CHECK:")
    print("Description (150-200 words):", 150 <= len(description.split()) <= 200)
    print("Features (5-7):", 5 <= len(features) <= 7)
    print("Keywords (10-15):", 10 <= len(keywords) <= 15)

except (json.JSONDecodeError, TypeError) as e:
    print("Invalid JSON response:", e)


VALIDATING AI RESPONSE
Valid JSON: True
Title: Turtle Check Men Navy Blue Casual Shirt
Description words: 151
Number of features: 7
Number of keywords: 14

REQUIREMENTS CHECK:
Description (150-200 words): True
Features (5-7): True
Keywords (10-15): True


In [20]:
def validate_listing(listing):
    description = listing.get("description", "")
    features = listing.get("features", [])
    keywords = listing.get("keywords", [])

    word_count = len(description.split())

    errors = []

    if not 150 <= word_count <= 200:
        errors.append(
            f"Description must have 150-200 words. Found: {word_count}"
        )

    if not 5 <= len(features) <= 7:
        errors.append("Features must contain 5-7 items")

    if not 10 <= len(keywords) <= 15:
        errors.append("Keywords must contain 10-15 items")

    return errors

print("\nAUTOMATIC VALIDATION")
print("=" * 50)

test_listing = {
    "description": "Example product description",
    "features": ["Feature 1", "Feature 2"],
    "keywords": ["Keyword 1"]
}

validation_errors = validate_listing(test_listing)

print("Errors detected:", len(validation_errors))

for error in validation_errors:
    print("-", error)


AUTOMATIC VALIDATION
Errors detected: 3
- Description must have 150-200 words. Found: 3
- Features must contain 5-7 items
- Keywords must contain 10-15 items


In [ ]:
# Validate the real AI-generated product listing

print("\nREAL PRODUCT VALIDATION")
print("=" * 50)

validation_errors = validate_listing(listing)

if not validation_errors:
    print("Product listing passed all validations!")
else:
    print("Product listing failed validation.")

    for error in validation_errors:
        print("-", error)

In [22]:
# Select the first three products
selected_products = dataset.select(range(3))

print("\nPRODUCTS SELECTED FOR BATCH PROCESSING")
print("=" * 50)

for product in selected_products:
    print("ID:", product["id"])
    print("Name:", product["productDisplayName"])
    print("Category:", product["masterCategory"])
    print("Image available:", product["image"] is not None)
    print("-" * 50)


PRODUCTS SELECTED FOR BATCH PROCESSING
ID: 15970
Name: Turtle Check Men Navy Blue Shirt
Category: Apparel
Image available: True
--------------------------------------------------
ID: 39386
Name: Peter England Men Party Blue Jeans
Category: Apparel
Image available: True
--------------------------------------------------
ID: 59263
Name: Titan Women Silver Watch
Category: Accessories
Image available: True
--------------------------------------------------


In [23]:
def create_product_prompt(product):

    prompt = f"""
You are an expert e-commerce copywriter.

Create a professional product listing using the product
information below and the product image.

PRODUCT INFORMATION:
Name: {product['productDisplayName']}
Category: {product['masterCategory']}
Subcategory: {product['subCategory']}
Type: {product['articleType']}
Color: {product['baseColour']}
Gender: {product['gender']}
Usage: {product['usage']}

REQUIREMENTS:
1. Create an attractive product title.
2. Write a detailed description between 150 and 200 words.
3. Include 5-7 key product features.
4. Generate 10-15 SEO keywords.
5. Use a professional and persuasive tone.
6. Do not invent materials or specifications.
7. Do not invent a price.
8. Return only valid JSON.

JSON STRUCTURE:
{{
    "title": "Product title",
    "description": "Product description",
    "features": ["Feature 1", "Feature 2"],
    "keywords": ["Keyword 1", "Keyword 2"]
}}
"""
    return prompt

# Test the function with the second product
test_prompt = create_product_prompt(dataset[1])

print("PROMPT FOR SECOND PRODUCT")
print("=" * 50)
print(test_prompt)

PROMPT FOR SECOND PRODUCT

You are an expert e-commerce copywriter.

Create a professional product listing using the product
information below and the product image.

PRODUCT INFORMATION:
Name: Peter England Men Party Blue Jeans
Category: Apparel
Subcategory: Bottomwear
Type: Jeans
Color: Blue
Gender: Men
Usage: Casual

REQUIREMENTS:
1. Create an attractive product title.
2. Write a detailed description between 150 and 200 words.
3. Include 5-7 key product features.
4. Generate 10-15 SEO keywords.
5. Use a professional and persuasive tone.
6. Do not invent materials or specifications.
7. Do not invent a price.
8. Return only valid JSON.

JSON STRUCTURE:
{
    "title": "Product title",
    "description": "Product description",
    "features": ["Feature 1", "Feature 2"],
    "keywords": ["Keyword 1", "Keyword 2"]
}



In [24]:
import base64
from io import BytesIO

def encode_product_image(product):
    # Get the product image
    image = product["image"]

    # Create a temporary memory buffer
    buffer = BytesIO()

    # Convert the image to PNG
    image.convert("RGB").save(buffer, format="PNG")

    # Encode the image to Base64
    base64_image = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    return base64_image

print("\nTESTING IMAGE ENCODING")
print("=" * 50)

for product in selected_products:
    encoded_image = encode_product_image(product)

    print("Product:", product["productDisplayName"])
    print("Base64 length:", len(encoded_image))
    print("-" * 50)


TESTING IMAGE ENCODING
Product: Turtle Check Men Navy Blue Shirt
Base64 length: 10856
--------------------------------------------------
Product: Peter England Men Party Blue Jeans
Base64 length: 8044
--------------------------------------------------
Product: Titan Women Silver Watch
Base64 length: 4224
--------------------------------------------------


In [27]:
import json
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_ADMIN_KEY"))

def generate_product_listing(product):

    # Step 1: Create the prompt
    prompt = create_product_prompt(product)

    # Step 2: Encode the image
    encoded_image = encode_product_image(product)

    # Step 3: Prepare the API message
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{encoded_image}"
                    }
                }
            ]
        }
    ]

    # Step 4: Call OpenAI
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            response_format={"type": "json_object"},
            max_tokens=1200
        )

        # Step 5: Convert the response to Python dictionary
        listing = json.loads(response.choices[0].message.content)

        return listing

    except Exception as e:
        print("Error generating listing:", type(e).__name__, str(e))
        return None

print("Product listing generator function created successfully!")

Product listing generator function created successfully!


In [28]:
from pathlib import Path

# Create folder for generated listings
output_dir = Path("generated_listings")
output_dir.mkdir(exist_ok=True)

# Process only the two remaining products
for product in selected_products.select(range(1, 3)):

    print("\nProcessing:", product["productDisplayName"])

    listing = generate_product_listing(product)

    if listing is not None:

        # Save each listing as a JSON file
        output_file = output_dir / f"product_{product['id']}.json"

        with open(output_file, "w", encoding="utf-8") as file:
            json.dump(listing, file, indent=4, ensure_ascii=False)

        print("Listing saved:", output_file)

    else:
        print("Failed to generate listing.")


Processing: Peter England Men Party Blue Jeans
Listing saved: generated_listings/product_39386.json

Processing: Titan Women Silver Watch
Listing saved: generated_listings/product_59263.json


In [29]:
def create_correction_prompt(listing):

    description = listing.get("description", "")

    prompt = f"""
You are an expert e-commerce copywriter.

The following product description is too short.

PRODUCT TITLE:
{listing.get('title')}

CURRENT DESCRIPTION:
{description}

TASK:
Rewrite and expand the description to between 160 and 180 words.

REQUIREMENTS:
1. Keep the original product information accurate.
2. Do not invent materials, specifications, prices or features.
3. Use a professional and persuasive e-commerce tone.
4. Avoid unnecessary repetition.
5. Return only the corrected description as plain text.
"""

    return prompt


print("Correction prompt function created successfully!")

import json

with open(
    "generated_listings/product_39386.json",
    "r",
    encoding="utf-8"
) as file:
    jeans_listing = json.load(file)

correction_prompt = create_correction_prompt(jeans_listing)

print("\nCORRECTION PROMPT FOR JEANS")
print("=" * 50)
print(correction_prompt)

Correction prompt function created successfully!

CORRECTION PROMPT FOR JEANS

You are an expert e-commerce copywriter.

The following product description is too short.

PRODUCT TITLE:
Peter England Men's Blue Party Jeans - Casual Elegance

CURRENT DESCRIPTION:
Step into effortless style with the Peter England Men's Party Blue Jeans. Crafted for the modern man, these jeans blend comfort with a sleek look, making them perfect for casual outings and special gatherings alike. The classic blue color offers versatility, allowing for easy pairing with your favorite shirts and footwear. With a tailored fit that complements all body types, these jeans feature just the right amount of stretch for ease of movement. Elevate your wardrobe and make a fashionable statement with these jeans that are designed to keep you looking sharp, no matter the occasion. Whether you're heading out for lunch or attending a party, these jeans are your go-to choice for casual sophistication.

TASK:
Rewrite and expan

In [30]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

# Load API key
load_dotenv(override=True)

api_key = os.getenv("OPENAI_ADMIN_KEY")

if not api_key:
    raise ValueError("API key not found")

client = OpenAI(api_key=api_key)

# Request corrected description
try:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": correction_prompt
            }
        ],
        max_tokens=600
    )

    corrected_description = response.choices[0].message.content.strip()

    print("\nCORRECTED JEANS DESCRIPTION")
    print("=" * 50)
    print(corrected_description)

    print("\nWord count:", len(corrected_description.split()))

except Exception as e:
    print("Correction failed!")
    print("Error:", type(e).__name__, str(e))


CORRECTED JEANS DESCRIPTION
Elevate your style with the Peter England Men's Blue Party Jeans, the perfect blend of casual elegance and comfort. Designed for the contemporary man, these jeans are an essential addition to any wardrobe. The rich blue hue offers versatile styling options, effortlessly pairing with everything from crisp button-downs to relaxed tees, making them ideal for both laid-back weekends and festive gatherings alike.

Crafted with a tailored fit that flatters all body types, these jeans ensure you look polished while maintaining freedom of movement. The subtle stretch in the fabric allows for all-day comfort, whether you're enjoying a casual lunch or mingling at a lively party.

Featuring thoughtful detailing and a timeless silhouette, these jeans embody sophistication without sacrificing ease. Make a statement with a garment that instills confidence and complements your unique style. Upgrade your collection today with the Peter England Men's Blue Party Jeans—a go-t

In [31]:
import json

# Verify the corrected description
word_count = len(corrected_description.split())

if 150 <= word_count <= 200:

    # Update only the description
    jeans_listing["description"] = corrected_description

    # Save the updated product listing
    with open(
        "generated_listings/product_39386.json",
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(jeans_listing, file, indent=4, ensure_ascii=False)

    print("Jeans listing updated successfully!")
    print("Description words:", word_count)

else:
    print("Description does not meet the requirements.")
    print("JSON file was not modified.")

Jeans listing updated successfully!
Description words: 175


In [32]:
import json

# Load the watch listing
with open(
    "generated_listings/product_59263.json",
    "r",
    encoding="utf-8"
) as file:
    watch_listing = json.load(file)

# Create the correction prompt
watch_correction_prompt = create_correction_prompt(watch_listing)

print("\nCORRECTION PROMPT FOR WATCH")
print("=" * 50)
print(watch_correction_prompt)


CORRECTION PROMPT FOR WATCH

You are an expert e-commerce copywriter.

The following product description is too short.

PRODUCT TITLE:
Elegant Titan Women's Silver Watch - Timeless Casual Accessory

CURRENT DESCRIPTION:
Introducing the Titan Women Silver Watch, a perfect blend of elegance and simplicity designed for the modern woman. This casual watch effortlessly complements any outfit, making it an essential accessory for both daily wear and special occasions. Featuring a sleek silver-tone finish and a minimalist design, the watch adds a touch of sophistication to your wrist. The easy-to-read dial ensures you stay on schedule, while the comfortable band allows for all-day wear. Durable and stylish, the Titan Women Silver Watch is not only a practical timepiece but also a statement of your unique style. Whether you’re headed to the office or a weekend brunch, this watch will elevate your look effortlessly.

TASK:
Rewrite and expand the description to between 160 and 180 words.

REQUI

In [33]:
# Correct the watch description using OpenAI

try:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": watch_correction_prompt
            }
        ],
        max_tokens=600
    )

    corrected_watch_description = (
        response.choices[0].message.content.strip()
    )

    print("\nCORRECTED WATCH DESCRIPTION")
    print("=" * 50)
    print(corrected_watch_description)

    print(
        "\nWord count:",
        len(corrected_watch_description.split())
    )

except Exception as e:
    print("Correction failed!")
    print("Error:", type(e).__name__, str(e))


CORRECTED WATCH DESCRIPTION
Discover the epitome of elegance with the Titan Women's Silver Watch, a timeless accessory crafted for the modern woman. This exquisite timepiece effortlessly marries sophistication and simplicity, making it a versatile addition to your jewelry collection. Ideal for both everyday wear and special occasions, the watch features a sleek silver-tone finish that complements any outfit—from professional attire to casual weekend ensembles.

Its minimalist design showcases an easy-to-read dial that keeps you on schedule without sacrificing style. The comfortable band ensures a perfect fit for all-day wear, allowing you to transition seamlessly from the boardroom to brunch with friends. Whether you’re attending a formal event or enjoying a leisurely day out, this watch adds a touch of refinement to your wrist.

Durable yet chic, the Titan Women Silver Watch is more than just a practical timepiece; it’s an expression of your unique style. Elevate your accessory game 

In [34]:
import json

# Count words in the corrected description
word_count = len(corrected_watch_description.split())

if 150 <= word_count <= 200:

    # Update only the watch description
    watch_listing["description"] = corrected_watch_description

    # Save the updated JSON
    with open(
        "generated_listings/product_59263.json",
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(watch_listing, file, indent=4, ensure_ascii=False)

    print("Watch listing updated successfully!")
    print("Description words:", word_count)

else:
    print("Description does not meet the requirements.")
    print("JSON file was not modified.")

Watch listing updated successfully!
Description words: 175
